# Corretor de Simulados — CASD

**Antes de rodar:** suba os cartões escaneados (`.tif`) na pasta `data/entrada`
do Drive Compartilhado. A célula 1 cria essa pasta se ela ainda não existir e
mostra o caminho exato.

**Depois:** rode as células 1 a 5 na ordem. A planilha final é salva
automaticamente em `data/resultados`, no Drive — não se perde ao fechar o Colab.

O código é baixado do GitHub a cada sessão, então roda sempre a versão mais
recente. Nada precisa ser atualizado no Drive à mão.

Runtime: **GPU T4** (Ambiente de execução → Alterar tipo de ambiente de execução).

In [ ]:
# @title 🔧 1. Preparar ambiente
# @markdown Monta o Drive, baixa o código do GitHub e prepara as pastas. Rodar uma vez por sessão.
!pip install -q opencv-python-headless torch torchvision xlsxwriter

import os, sys, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

# No Drive ficam apenas os dados. O codigo vem do GitHub.
DRIVE      = Path(r"/content/drive/Shareddrives/Departamentos Externos/Financeiro & Jurídico/CORRETOR DE SIMULADOS")
ENTRADA    = DRIVE / "data" / "entrada"
RESULTADOS = DRIVE / "data" / "resultados"
ENTRADA.mkdir(parents=True, exist_ok=True)
RESULTADOS.mkdir(parents=True, exist_ok=True)

# Clone raso (--depth 1): baixa so a versao atual, sem o historico. Rapido.
CODIGO = Path("/content/corretor-de-simulados")

# Sai da pasta antes de apaga-la: numa segunda execucao o Colab esta dentro dela,
# e o rmtree deixaria o processo sem diretorio de trabalho.
os.chdir("/content")

if CODIGO.exists():
    shutil.rmtree(CODIGO)
!git clone --depth 1 -q https://github.com/CASD-curso/corretor-de-simulados.git "{CODIGO}"

if not (CODIGO / "corretor" / "config.py").exists():
    raise RuntimeError("Falha ao baixar o codigo do GitHub. Confira a conexao e rode de novo.")

os.chdir(CODIGO)
if str(CODIGO) not in sys.path:
    sys.path.insert(0, str(CODIGO))

print("Codigo baixado em :", CODIGO)
print("Suba os cartoes em:", ENTRADA)
print("Planilhas sairao em:", RESULTADOS)

In [ ]:
# @title 📥 2. Copiar os cartões do Drive
# @markdown Leva os `.tif` para o disco local do Colab, que é muito mais rápido que o Drive montado.
import glob

destino = "/content/dados_locais/originais"
if os.path.exists(destino):
    shutil.rmtree(destino)
os.makedirs(destino, exist_ok=True)

for arq in glob.glob(str(ENTRADA / "*.tif*")):
    shutil.copy2(arq, destino)

qtd = len(os.listdir(destino))
print(f"Cartoes copiados: {qtd}")
if qtd == 0:
    print(f"\nNenhum .tif encontrado. Suba os cartoes escaneados em:\n   {ENTRADA}")
else:
    print("\nSe o numero estiver diferente do esperado e voce tem certeza do upload,")
    print("e cache do Drive. Aguarde alguns minutos e rode esta celula de novo.")

In [ ]:
# @title ⚙️ 3. Configurar o simulado
# @markdown Nome do arquivo de saída, modelo da prova e gabarito oficial.

nome_simulado = "SEMI 1"  # @param {type:"string"}
simulado_selecionado = "SEMI"  # @param ["CASDINHO", "SEMI"]
gabarito_oficial = ""  # @param {type:"string"}

os.makedirs('/content/dados_locais', exist_ok=True)
gabarito_oficial = gabarito_oficial.upper().replace(" ", "").strip()

esperado = 50 if simulado_selecionado == "CASDINHO" else 60
if len(gabarito_oficial) != esperado:
    raise ValueError(
        f"O gabarito de {simulado_selecionado} precisa de {esperado} letras. "
        f"Voce digitou {len(gabarito_oficial)}.")

for arquivo, conteudo in [
    ('nome_simulado.txt', nome_simulado.strip()),
    ('simulado_ativo.txt', simulado_selecionado),
    ('gabarito_atual.txt', gabarito_oficial),
]:
    with open(f'/content/dados_locais/{arquivo}', 'w', encoding='utf-8') as f:
        f.write(conteudo)

# O config le esses arquivos no momento em que e importado, e o Python guarda
# modulos ja importados em cache. Descarregamos os do projeto para que uma
# mudanca de gabarito aqui valha de verdade nas celulas seguintes.
for modulo in [m for m in list(sys.modules) if m.startswith("corretor")]:
    del sys.modules[modulo]

print(f"Modelo: {simulado_selecionado} ({esperado} questoes)")
print(f"Saida : {nome_simulado}.xlsx")
print("Gabarito registrado.")

In [ ]:
# @title ✂️ 4. Processar os cartões
# @markdown Alinha cada folha, recorta as bolhas e as prepara para a leitura.
from corretor.visao.extracao_em_lote import processar_simulados

processar_simulados(extrair_respostas=True, extrair_inscricao=True)

In [ ]:
# @title 🔎 4.5. Gerar planilha de revisão (opcional)
# @markdown Mostra o que precisa de conferência antes do cálculo da nota.
# @markdown Quem não tiver nada a corrigir pode pular direto pra célula 5.
from corretor.revisao.gerar_planilha_revisao import gerar_planilha_revisao
from corretor.config import PLANILHA_REVISAO_PATH
from google.colab import files

_, LINHAS_ORIGINAIS = gerar_planilha_revisao()
print("\nBaixe, revise a aba 'Correção' (e a seção de alinhamento na aba")
print("'Checkup', se houver) e suba de volta na célula seguinte.")
files.download(str(PLANILHA_REVISAO_PATH))

In [ ]:
# @title 📤 4.6. Subir a planilha revisada (só se revisou)
# @markdown Sem upload aqui, a célula 5 calcula direto da leitura automática.
from corretor.revisao.aplicar_revisao import aplicar_revisao
from corretor.inferencia.inferir_completo import gerar_planilha_unificada
from google.colab import files

_upload = files.upload()
if _upload:
    caminho_revisado = list(_upload.keys())[0]
    linhas_originais = globals().get("LINHAS_ORIGINAIS")
    if linhas_originais is None:
        # Fallback: a célula 4.5 não rodou nesta sessão do Colab (ex.: reiniciou
        # o runtime só até aqui). Sem ela, precisamos recalcular do zero.
        print("Aviso: célula 4.5 não rodou nesta sessão — recalculando a leitura automática.")
        linhas_originais, _, _, _ = gerar_planilha_unificada()
    DADOS_REVISADOS = aplicar_revisao(caminho_revisado, linhas_originais)
    print(f"\n{len(DADOS_REVISADOS)} simulado(s) prontos, com a revisão aplicada.")
else:
    DADOS_REVISADOS = None
    print("Nada subido — a célula 5 vai calcular direto da leitura automática.")

In [ ]:
# @title 📊 5. Gerar a planilha e salvar no Drive
from corretor.relatorio.gerar_excel import gerar_excel_final
from corretor.config import OUTPUTS_DIR, NOME_SIMULADO

gerar_excel_final(dados_brutos=globals().get("DADOS_REVISADOS"))

origem = Path(OUTPUTS_DIR) / f"{NOME_SIMULADO}.xlsx"
if not origem.exists():
    raise FileNotFoundError(f"A planilha nao foi gerada em {origem}.")

final = RESULTADOS / origem.name
shutil.copy2(origem, final)
print(f"\nPlanilha salva no Drive:\n   {final}")

---
## Diagnóstico (opcional)

Só rodar se algo der errado.

In [ ]:
# @title 🖥️ Ambiente: GPU e dependências
print("Runtime\n" + "-"*46)
try:
    import torch, torchvision
    print(f"torch       {torch.__version__}")
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"GPU         {p.name} ({p.total_memory/1e9:.1f} GB)")
    else:
        print("GPU         nenhuma - roda em CPU, so um pouco mais devagar")
except ImportError as e:
    print("torch AUSENTE:", e)

print("\nDependencias\n" + "-"*46)
for nome, mod in [("opencv","cv2"),("pandas","pandas"),("numpy","numpy"),
                  ("Pillow","PIL"),("xlsxwriter","xlsxwriter")]:
    try:
        m = __import__(mod)
        print(f"{nome:<12} {getattr(m,'__version__','ok')}")
    except ImportError:
        print(f"{nome:<12} AUSENTE")

# Medido fora do Colab (CPU, 1 folha SEMI = 370 bolhas): extracao ~320 ms,
# inferencia ~360 ms, dos quais so ~23% e a rede. A T4 encosta em ~12% do
# tempo total - o gargalo e o laco que processa uma bolha por vez.

In [ ]:
# @title 📁 Estrutura: os caminhos batem?
from pathlib import Path
from corretor.config import (SIMULADO_ATIVO, NUM_QUESTOES, NOME_SIMULADO,
                             SIMULADOS_DIR, CNN_BOLHAS_PATH, LOCAL_DIR,
                             RECORTES_BOLHAS_DIR, RECORTES_INSCRICAO_DIR)

print(f"Simulado ativo: {SIMULADO_ATIVO} ({NUM_QUESTOES} questoes)")
print(f"Saida         : {NOME_SIMULADO}.xlsx\n")

for nome, p in {
    "cartoes (.tif)":     SIMULADOS_DIR,
    "pesos da rede":      CNN_BOLHAS_PATH,
    "gabarito_atual.txt": LOCAL_DIR / "gabarito_atual.txt",
    "entrada no Drive":   ENTRADA,
    "resultados no Drive":RESULTADOS,
}.items():
    print(f"{'OK   ' if Path(p).exists() else 'FALTA'} {nome:<20} {p}")

tifs = list(Path(SIMULADOS_DIR).glob("*.tif*")) if Path(SIMULADOS_DIR).exists() else []
rec = (len(list(Path(RECORTES_BOLHAS_DIR).glob('*.png'))) +
       len(list(Path(RECORTES_INSCRICAO_DIR).glob('*.png')))) if Path(RECORTES_BOLHAS_DIR).exists() else 0
print(f"\nCartoes: {len(tifs)} | recortes gerados: {rec}")
if tifs:
    print(f"Esperado: ~{(NUM_QUESTOES*5+70)*len(tifs)} recortes ({NUM_QUESTOES*5+70} por folha)")

In [ ]:
# @title 🔍 Ver bolhas e probabilidades
# @markdown Use quando uma resposta sair errada: mostra o que o modelo viu.
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from corretor.config import RECORTES_BOLHAS_DIR, LIMIAR_MAXIMO, LIMIAR_DUPLA
from corretor.inferencia.modelo_bolhas import carregar_modelo, obter_device, prob_bolha_preenchida

QUESTAO = "Q1"  # @param {type:"string"}

# ATENCAO A INVERSAO: o ImageFolder ordena as classes em ordem alfabetica
# (preenchida=0, vazia=1), entao a sigmoid devolve a probabilidade de estar
# VAZIA. Probabilidade BAIXA = bolha marcada.
print(f"LIMIAR_MAXIMO={LIMIAR_MAXIMO} (menor prob acima disso: EM BRANCO)")
print(f"LIMIAR_DUPLA ={LIMIAR_DUPLA} (2a menor abaixo disso: NULA)\n")

device = obter_device(); modelo = carregar_modelo(device)
arqs = sorted(Path(RECORTES_BOLHAS_DIR).glob(f"*_{QUESTAO}_?.png"))[:10]
if not arqs:
    arqs = sorted(Path(RECORTES_BOLHAS_DIR).glob("*.png"))[:10]
    print(f"Nada para {QUESTAO}; mostrando as 10 primeiras.")

fig, axs = plt.subplots(2, 5, figsize=(13, 6))
for ax in axs.flat: ax.axis("off")
for ax, p in zip(axs.flat, arqs):
    pr, _leitura_ok = prob_bolha_preenchida(modelo, p, device)
    ax.imshow(Image.open(p), cmap="gray")
    ax.set_title(f"{p.stem.split('_')[-1]}\nprob={pr:.3f}",
                 color=("red" if pr < 0.5 else "black"), fontsize=10)
plt.suptitle("vermelho = preenchida (prob baixa)")
plt.tight_layout(); plt.show()